# tensor-unbind — worked example 2: Unbind along time axis to iterate over sequence steps

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-unbind`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`torch.unbind(x, dim=0)` (the default) peels a `(T, ...)` tensor into T separate tensors along the time axis. This is useful for processing a sequence step-by-step without manually indexing `x[0]`, `x[1]`, etc. Each resulting tensor is a view sharing the original storage.

## Worked solution

**Step 1 — Input shape.**
A trajectory tensor of shape `(T, B, D)` has T timesteps, B batch items, and D features. We want to process each timestep separately.

**Step 2 — Unbind along dim=0.**
`steps = t.unbind(traj, dim=0)` gives a tuple of T tensors, each of shape `(B, D)`.

**Step 3 — Iterate over the tuple.**
`for step_tensor in steps:` is equivalent to `for i in range(T): step_tensor = traj[i]` but cleaner and slightly faster (avoids repeated indexing overhead).

**Step 4 — Use case in RNNs.**
Many sequence models loop over timesteps. Unbinding the input once at the start is cleaner than re-indexing inside the loop.

In [ ]:
import torch as t

t.manual_seed(42)
T, B, D = 5, 3, 4
traj = t.randn(T, B, D)

# Unbind along time axis (default dim=0)
steps = t.unbind(traj, dim=0)
print('Number of steps:', len(steps))      # 5
print('Each step shape:', steps[0].shape)  # (3, 4)

# Process each step (example: compute per-step mean norm)
step_norms = []
for i, step in enumerate(steps):
    norm = step.norm(dim=-1).mean()  # mean L2 norm across batch
    step_norms.append(norm.item())
    print(f'  step {i}: mean_norm={norm.item():.4f}')

# Verify: each step matches direct indexing
for i in range(T):
    assert t.allclose(steps[i], traj[i]), f'step {i} mismatch'
print('All steps match direct indexing: True')